# Global Jazz Diasporas
### Map migration networks
This script creates geospatial data for Cisco Bradley's Global Jazz Diasporas project at the Pratt Institute Music & Migration Lab. It creates network map features for each musician-migration event. 

### Set up for replication
This script was designed to be run in ArcGIS Pro 3.5 using Python 3. To replicate, create a new ArcGIS project, add this notebook, and then run as follows. 

In [1]:
import arcpy
import os
import pandas
import geopandas
import warnings

In [2]:
#set up workspaces and check database
aprx = arcpy.mp.ArcGISProject("Current")
default_gdb = aprx.defaultGeodatabase
default_folder = aprx.homeFolder
arcpy.env.overwriteOutput = True
print("Directory: " + default_folder)
print("Geodatabase: " + default_gdb)

Directory: C:\Users\johnl\My Drive (jlauerma@pratt.edu)\Research\Music and Migration
Geodatabase: C:\Users\johnl\My Drive (jlauerma@pratt.edu)\Research\Music and Migration\MusicMigration.gdb


### Mapping the data
The next step is to geocode the tabular data into `origins` points, `destinations` points, `paths` lines (which show an individual migration path), and `lifepaths` lines (which connect all of a musician's individual migration paths into one line). The workflow is designed to use the [ESRI World Geocoding Serivce](https://www.arcgis.com/home/item.html?id=305f2e55e67f4389bef269669fc2e284). 

In [3]:
# Load the CSV to geodatabase 
csv = os.path.join(default_folder, "data_cleaned.csv")
arcpy.conversion.TableToGeodatabase(csv, default_gdb)
print(f"loaded data to {default_gdb}")

loaded data to C:\Users\johnl\My Drive (jlauerma@pratt.edu)\Research\Music and Migration\MusicMigration.gdb


In [4]:
#geocode origins
arcpy.geocoding.GeocodeAddresses(
    in_table=os.path.join(default_gdb, "data_cleaned"),
    address_locator="https://geocode.arcgis.com/arcgis/rest/services/World/GeocodeServer/Esri World Geocoder",
    in_address_fields="'Address or Place' <None> VISIBLE NONE;Address2 <None> VISIBLE NONE;Address3 <None> VISIBLE NONE;Neighborhood <None> VISIBLE NONE;City Depart_City VISIBLE NONE;County Depart_County VISIBLE NONE;State Depart_State VISIBLE NONE;ZIP <None> VISIBLE NONE;ZIP4 <None> VISIBLE NONE;Country Depart_Country VISIBLE NONE",
    out_feature_class=os.path.join(default_gdb, "origins"),
    out_relationship_type="STATIC",
    country="AFG;ALB;DZA;ASM;AND;AGO;AIA;ATA;ATG;ARG;ARM;ABW;AUS;AUT;AZE;BHS;BHR;BGD;BRB;BLR;BEL;BLZ;BEN;BMU;BTN;BOL;BES;BIH;BWA;BVT;BRA;IOT;VGB;BRN;BGR;BFA;BDI;CPV;KHM;CMR;CAN;CYM;CAF;TCD;CHL;CHN;CXR;CCK;COL;COM;COK;CRI;HRV;CUB;CUW;CYP;CZE;CIV;COD;DNK;DJI;DMA;DOM;ECU;EGY;SLV;GNQ;ERI;EST;SWZ;ETH;EUR;FLK;FRO;FSM;FJI;FIN;FRA;GUF;PYF;ATF;GAB;GMB;GEO;DEU;GHA;GIB;GRC;GRL;GRD;GLP;GUM;GTM;GGY;GIN;GNB;GUY;HTI;HMD;HND;HKG;HUN;ISL;IND;IDN;IRN;IRQ;IRL;IMN;ISR;ITA;JAM;JPN;JEY;JOR;KAZ;KEN;KIR;KWT;KGZ;LAO;LVA;LBN;LSO;LBR;LBY;LIE;LTU;LUX;MAC;MDG;MWI;MYS;MDV;MLI;MLT;MHL;MTQ;MRT;MUS;MYT;MEX;MDA;MCO;MNG;MNE;MSR;MAR;MOZ;MMR;NAM;NRU;NPL;NLD;NCL;NZL;NIC;NER;NGA;NIU;NFK;PRK;MKD;MNP;NOR;OMN;PAK;PLW;PSE;PAN;PNG;PRY;PER;PHL;PCN;POL;PRT;PRI;QAT;COG;REU;ROU;RUS;RWA;BLM;SHN;KNA;LCA;MAF;SPM;VCT;WSM;SMR;STP;SAU;SEN;SRB;SYC;SLE;SGP;SXM;SVK;SVN;SLB;SOM;ZAF;SGS;KOR;SSD;ESP;LKA;SDN;SUR;SJM;SWE;CHE;SYR;TWN;TJK;TZA;THA;TLS;TGO;TKL;TON;TTO;TUN;TKM;TCA;TUV;TUR;VIR;UGA;UKR;ARE;GBR;USA;UMI;URY;UZB;VUT;VAT;VEN;VNM;WLF;ESH;YEM;ZMB;ZWE",
    location_type="ADDRESS_LOCATION",
    category=None,
    output_fields="LOCATION_ONLY"
)

## print to verify
results = arcpy.management.GetCount(os.path.join(default_gdb, "origins"))
count = int(results[0])
print(f"geocoded {count} origin points")

geocoded 15695 origin points


In [6]:
# view to verify
warnings.filterwarnings("ignore")
origins_gdf = geopandas.read_file(default_gdb, layer='origins')
origins_gdf.head(50)

,SegmentID,MusicianID,Musician_Name,OriginID,DestinationID,Arrival_State,Arrival_County,Arrival_City,Surname,Depart,...,Ragtime,Salsa,Soul_Jazz,Smooth_Jazz,Soul,Stride_Piano,Swing,Vaudeville,World_Fusion,geometry
0,10001839nan,1000,"Harper, Elijah ""Buddy""",nan_1000_nan,Atlanta_1000_1839,Georgia,Fulton,Atlanta,Harper,None,...,0,0,0,0,0,0,0,0,0,POINT (NaN NaN)
1,100116801700,1001,"Harrigan, Eric",Frederiksted_1001_1700,Marigot_1001_1680,St. Martin,Marigot,Marigot,None,None,...,0,0,0,0,0,0,0,0,0,POINT (-64.88258 17.71223)
2,100117031734,1001,"Harrigan, Eric",Frederiksted_1001_1734,nan_1001_1703,Nevis,None,None,None,None,...,0,0,0,0,0,0,0,0,0,POINT (-64.88258 17.71223)
3,100117501821,1001,"Harrigan, Eric",nan_1001_1821,Spanish Town_1001_1750,Virgin Islands,Virgin Gorda,Spanish Town,None,None,...,0,0,0,0,0,0,0,0,0,POINT (NaN NaN)
4,100018491895,1000,"Harper, Elijah ""Buddy""",Atlanta_1000_1895,nan_1000_1849,Virginia,None,None,Blackman,None,...,0,0,0,0,0,0,0,0,0,POINT (-84.3915 33.74855)
5,100118211861,1001,"Harrigan, Eric",nan_1001_1861,nan_1001_1821,Anegada,None,None,None,None,...,0,0,0,0,0,0,0,0,0,POINT (-64.81909 17.72811)
6,100118611893,1001,"Harrigan, Eric",nan_1001_1893,nan_1001_1861,Virgin Islands,Virgin Gorda,None,None,None,...,0,0,0,0,0,0,0,0,0,POINT (NaN NaN)
7,100118681950,1001,"Harrigan, Eric",St. Thomas_1001_1950,nan_1001_1868,Virgin Islands,Tortola,None,None,None,...,0,0,0,0,0,0,0,0,0,POINT (-64.9307 18.3419)
8,100119111942,1001,"Harrigan, Eric",nan_1001_1942,Charlotte Amalie_1001_1911,St. Thomas and St. John,St. Thomas,Charlotte Amalie,None,None,...,0,0,0,0,0,0,0,0,0,POINT (NaN NaN)
9,019621965,0,None,Middletown_0_1965,Paris_0_1962,Ile-de-France,Paris,Paris,None,None,...,0,0,0,0,0,0,0,0,0,POINT (-72.65065 41.56232)


In [7]:
#geocode destinations
arcpy.geocoding.GeocodeAddresses(
    in_table=os.path.join(default_gdb, "data_cleaned"),
    address_locator="https://geocode.arcgis.com/arcgis/rest/services/World/GeocodeServer/Esri World Geocoder",
    in_address_fields="'Address or Place' <None> VISIBLE NONE;Address2 <None> VISIBLE NONE;Address3 <None> VISIBLE NONE;Neighborhood <None> VISIBLE NONE;City Arrival_City VISIBLE NONE;County Arrival_County VISIBLE NONE;State Arrival_State VISIBLE NONE;ZIP <None> VISIBLE NONE;ZIP4 <None> VISIBLE NONE;Country Arrival_Country VISIBLE NONE",
    out_feature_class=os.path.join(default_gdb, "destinations"),
    out_relationship_type="STATIC",
    country="AFG;ALB;DZA;ASM;AND;AGO;AIA;ATA;ATG;ARG;ARM;ABW;AUS;AUT;AZE;BHS;BHR;BGD;BRB;BLR;BEL;BLZ;BEN;BMU;BTN;BOL;BES;BIH;BWA;BVT;BRA;IOT;VGB;BRN;BGR;BFA;BDI;CPV;KHM;CMR;CAN;CYM;CAF;TCD;CHL;CHN;CXR;CCK;COL;COM;COK;CRI;HRV;CUB;CUW;CYP;CZE;CIV;COD;DNK;DJI;DMA;DOM;ECU;EGY;SLV;GNQ;ERI;EST;SWZ;ETH;EUR;FLK;FRO;FSM;FJI;FIN;FRA;GUF;PYF;ATF;GAB;GMB;GEO;DEU;GHA;GIB;GRC;GRL;GRD;GLP;GUM;GTM;GGY;GIN;GNB;GUY;HTI;HMD;HND;HKG;HUN;ISL;IND;IDN;IRN;IRQ;IRL;IMN;ISR;ITA;JAM;JPN;JEY;JOR;KAZ;KEN;KIR;KWT;KGZ;LAO;LVA;LBN;LSO;LBR;LBY;LIE;LTU;LUX;MAC;MDG;MWI;MYS;MDV;MLI;MLT;MHL;MTQ;MRT;MUS;MYT;MEX;MDA;MCO;MNG;MNE;MSR;MAR;MOZ;MMR;NAM;NRU;NPL;NLD;NCL;NZL;NIC;NER;NGA;NIU;NFK;PRK;MKD;MNP;NOR;OMN;PAK;PLW;PSE;PAN;PNG;PRY;PER;PHL;PCN;POL;PRT;PRI;QAT;COG;REU;ROU;RUS;RWA;BLM;SHN;KNA;LCA;MAF;SPM;VCT;WSM;SMR;STP;SAU;SEN;SRB;SYC;SLE;SGP;SXM;SVK;SVN;SLB;SOM;ZAF;SGS;KOR;SSD;ESP;LKA;SDN;SUR;SJM;SWE;CHE;SYR;TWN;TJK;TZA;THA;TLS;TGO;TKL;TON;TTO;TUN;TKM;TCA;TUV;TUR;VIR;UGA;UKR;ARE;GBR;USA;UMI;URY;UZB;VUT;VAT;VEN;VNM;WLF;ESH;YEM;ZMB;ZWE",
    location_type="ADDRESS_LOCATION",
    category=None,
    output_fields="LOCATION_ONLY"
)

## print to verify
results = arcpy.management.GetCount(os.path.join(default_gdb, "destinations"))
count = int(results[0])
print(f"geocoded {count} destination points")

geocoded 15695 destination points


In [8]:
# view to verify
destinations_gdf = geopandas.read_file(default_gdb, layer='destinations')
destinations_gdf.head(50)

,SegmentID,MusicianID,Musician_Name,OriginID,DestinationID,Arrival_State,Arrival_County,Arrival_City,Surname,Depart,...,Ragtime,Salsa,Soul_Jazz,Smooth_Jazz,Soul,Stride_Piano,Swing,Vaudeville,World_Fusion,geometry
0,100018491895,1000,"Harper, Elijah ""Buddy""",Atlanta_1000_1895,nan_1000_1849,Virginia,None,None,Blackman,None,...,0,0,0,0,0,0,0,0,0,POINT (-78.69795 37.51282)
1,100117001750,1001,"Harrigan, Eric",Spanish Town_1001_1750,Frederiksted_1001_1700,St. Croix,Frederiksted,Frederiksted,None,None,...,0,0,0,0,0,0,0,0,0,POINT (-64.88179 17.71233)
2,019621965,0,None,Middletown_0_1965,Paris_0_1962,Ile-de-France,Paris,Paris,None,None,...,0,0,0,0,0,0,0,0,0,POINT (2.35253 48.85637)
3,100117031734,1001,"Harrigan, Eric",Frederiksted_1001_1734,nan_1001_1703,Nevis,None,None,None,None,...,0,0,0,0,0,0,0,0,0,POINT (-62.58664 17.1506)
4,100118211861,1001,"Harrigan, Eric",nan_1001_1861,nan_1001_1821,Anegada,None,None,None,None,...,0,0,0,0,0,0,0,0,0,POINT (NaN NaN)
5,100116801700,1001,"Harrigan, Eric",Frederiksted_1001_1700,Marigot_1001_1680,St. Martin,Marigot,Marigot,None,None,...,0,0,0,0,0,0,0,0,0,POINT (-61.01368 13.96735)
6,10001839nan,1000,"Harper, Elijah ""Buddy""",nan_1000_nan,Atlanta_1000_1839,Georgia,Fulton,Atlanta,Harper,None,...,0,0,0,0,0,0,0,0,0,POINT (-84.3915 33.74855)
7,100118681950,1001,"Harrigan, Eric",St. Thomas_1001_1950,nan_1001_1868,Virgin Islands,Tortola,None,None,None,...,0,0,0,0,0,0,0,0,0,POINT (-64.81909 17.72811)
8,100118611893,1001,"Harrigan, Eric",nan_1001_1893,nan_1001_1861,Virgin Islands,Virgin Gorda,None,None,None,...,0,0,0,0,0,0,0,0,0,POINT (-64.81909 17.72811)
9,10018141839,100,"Ashby, Harold",nan_100_1839,nan_100_1814,Virginia,None,None,Ashby,None,...,0,0,0,0,0,0,1,0,0,POINT (-78.69795 37.51282)


### Draw the network
After geocoding the origin and destination points, we draw a linear network across them. This workflow will create a `paths` layer where each line indicates a single migration event (e.g. a musician moved from point A to point B) and a `life_paths` layer that combines all migration events into the lifecourse of each musician. 

In [9]:
#merge the layers into one point layer
##define the new layers
origins_layer = os.path.join(default_gdb, "origins")
destinations_layer = os.path.join(default_gdb, "destinations")

## Add a LocationType field in origins
arcpy.management.AddField(origins_layer, "LocationType", "TEXT")
arcpy.management.CalculateField(origins_layer, "LocationType", '"Origin"', "PYTHON3")

##Calculate the LocationType field for origins and destinations
arcpy.management.AddField(destinations_layer, "LocationType", "TEXT")
arcpy.management.CalculateField(destinations_layer, "LocationType", '"Destination"', "PYTHON3")

##merge
arcpy.management.Merge(
    inputs="origins;destinations",
    output="origins_destinations_merged",
    field_mappings=None,
    add_source="NO_SOURCE_INFO",
    field_match_mode="AUTOMATIC"
)

## print to verify
results = arcpy.management.GetCount(os.path.join(default_gdb, "origins_destinations_merged"))
count = int(results[0])
print(f"create a point layer with {count} origins and/or destinations")

create a point layer with 31390 origins and/or destinations


In [11]:
#create path segments between origins and destinations
arcpy.management.PointsToLine(
    Input_Features="origins_destinations_merged",
    Output_Feature_Class="paths",
    Line_Field="SegmentID",
    Sort_Field=None,
    Close_Line="NO_CLOSE",
    Line_Construction_Method="TWO_POINT",
    Attribute_Source="START",
    Transfer_Fields="SegmentID;MusicianID;Musician_Name;OriginID;DestinationID;Arrival_State;Arrival_County;Arrival_City;Surname;Depart;Arrival;Depart_1;Depart_Country;Depart_State;Depart_County;Depart_City;Final_Destination;Genre;Instrument;Free;Dyn;Public;Dynasty_Name;All_Genres;Afro_Cuban_Jazz;African_Jazz;Acid_Jazz;Acid_Rock;Bebop;Boogaloo;Blues;Bossa_Nova;Broadway;Brass_Band;Boogie_Woogie;Spirituals;Circus;European_Classical;Folk_Cape_Verde;Country;Disco;Doo_Wop;Early_Jazz;Easy_Listening;Free_Jazz;Funk;Folk_American;Fusion;General_Jazz;Gospel;Hard_Bop;Hip_Hop;House;Jump_Blues;Jazz_Funk;Latin_Jazz;Latin;Military_Band;Minstrelsy;Motown;Musical_Theater;New_Wave;Opera;Pop;Rhythm_and_Blues;Rock;Ragtime;Salsa;Soul_Jazz;Smooth_Jazz;Soul;Stride_Piano;Swing;Vaudeville;World_Fusion;LocationType"
)
    
## print to verify
results = arcpy.management.GetCount(os.path.join(default_gdb, "paths"))
count = int(results[0])
print(f"created a layer of {count} migration paths")

created a layer of 15237 migration paths


In [12]:
# check structure
paths_gdf = geopandas.read_file(default_gdb, layer='paths')
paths_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 15237 entries, 0 to 15236
Data columns (total 79 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   Shape_Length              15237 non-null  float64 
 1   SegmentID                 15237 non-null  object  
 2   START_SegmentID           15237 non-null  object  
 3   START_MusicianID          15237 non-null  int64   
 4   START_Musician_Name       15236 non-null  object  
 5   START_OriginID            15237 non-null  object  
 6   START_DestinationID       15237 non-null  object  
 7   START_Arrival_State       15237 non-null  object  
 8   START_Arrival_County      11350 non-null  object  
 9   START_Arrival_City        9343 non-null   object  
 10  START_Surname             14466 non-null  object  
 11  START_Depart              119 non-null    object  
 12  START_Arrival             15237 non-null  int64   
 13  START_Depart_1            15237 non-nu

In [13]:
# remove a naming convention that the PointsToLine tool adds...even though it has parameters that should stop it from doing that...

# List the field names
layer_name = "paths"
existing_fields = [field.name for field in arcpy.ListFields(layer_name)]

# Loop through fields. Change names and aliases.
for field in arcpy.ListFields(layer_name):
    if field.name.startswith("START_"):
        new_name = field.name.replace("START_", "")
        new_alias = field.aliasName.replace("START_", "") if field.aliasName.startswith("START_") else field.aliasName
        
        if new_name not in existing_fields:  # Avoid an error related to renaming existing fields
            arcpy.management.AlterField(layer_name, field.name, new_name, new_alias)
        else:
            print(f"Skipped {field.name} because {new_name} already exists.")

print("Field names and aliases updated.")

Skipped START_SegmentID because SegmentID already exists.
Field names and aliases updated.


In [14]:
# view to verify
paths_gdf = geopandas.read_file(default_gdb, layer='paths')
paths_gdf.head(50)

,Shape_Length,SegmentID,START_SegmentID,MusicianID,Musician_Name,OriginID,DestinationID,Arrival_State,Arrival_County,Arrival_City,...,Salsa,Soul_Jazz,Smooth_Jazz,Soul,Stride_Piano,Swing,Vaudeville,World_Fusion,LocationType,geometry
0,75.357018,019621965,019621965,0,None,Middletown_0_1965,Paris_0_1962,Ile-de-France,Paris,Paris,...,0,0,0,0,0,0,0,0,Origin,"MULTILINESTRING ((-72.65065 41.56232, 2.35253 ..."
1,6.825416,100018491895,100018491895,1000,"Harper, Elijah ""Buddy""",Atlanta_1000_1895,nan_1000_1849,Virginia,None,None,...,0,0,0,0,0,0,0,0,Origin,"MULTILINESTRING ((-84.3915 33.74855, -78.69795..."
2,5.384475,100116801700,100116801700,1001,"Harrigan, Eric",Frederiksted_1001_1700,Marigot_1001_1680,St. Martin,Marigot,Marigot,...,0,0,0,0,0,0,0,0,Origin,"MULTILINESTRING ((-64.88258 17.71223, -61.0136..."
3,2.363645,100117031734,100117031734,1001,"Harrigan, Eric",Frederiksted_1001_1734,nan_1001_1703,Nevis,None,None,...,0,0,0,0,0,0,0,0,Origin,"MULTILINESTRING ((-64.88258 17.71223, -62.5866..."
4,0.623851,100118681950,100118681950,1001,"Harrigan, Eric",St. Thomas_1001_1950,nan_1001_1868,Virgin Islands,Tortola,None,...,0,0,0,0,0,0,0,0,Origin,"MULTILINESTRING ((-64.9307 18.3419, -64.81909 ..."
5,0.623851,100118931911,100118931911,1001,"Harrigan, Eric",Charlotte Amalie_1001_1911,nan_1001_1893,Virgin Islands,Tortola,None,...,0,0,0,0,0,0,0,0,Origin,"MULTILINESTRING ((-64.9307 18.3419, -64.81909 ..."
6,14.404509,10018141839,10018141839,100,"Ashby, Harold",nan_100_1839,nan_100_1814,Virginia,None,None,...,0,0,0,0,0,1,0,0,Origin,"MULTILINESTRING ((-92.96262 39.51508, -78.6979..."
7,14.404509,10018171842,10018171842,100,"Ashby, Harold",nan_100_1842,nan_100_1817,Virginia,None,None,...,0,0,0,0,0,1,0,0,Origin,"MULTILINESTRING ((-92.96262 39.51508, -78.6979..."
8,14.404509,10018181840,10018181840,100,"Ashby, Harold",nan_100_1840,nan_100_1818,Virginia,None,None,...,0,0,0,0,0,1,0,0,Origin,"MULTILINESTRING ((-92.96262 39.51508, -78.6979..."
9,7.928251,10018221842,10018221842,100,"Ashby, Harold",nan_100_1842,nan_100_1822,Kentucky,None,None,...,0,0,0,0,0,1,0,0,Origin,"MULTILINESTRING ((-92.96262 39.51508, -85.2876..."


In [15]:
#create path segements to follow life journeys of musicians
arcpy.analysis.PairwiseDissolve(
    in_features="paths",
    out_feature_class="life_paths",
    dissolve_field="MusicianID",
    statistics_fields="Musician_Name FIRST;Final_Destination FIRST;Genre FIRST;All_Genres FIRST;Instrument FIRST;Free FIRST;Dyn FIRST;Dynasty_Name FIRST",
    multi_part="MULTI_PART",
    concatenation_separator=""
)

## print to verify
results = arcpy.management.GetCount(os.path.join(default_gdb, "origins"))
count = int(results[0])
print(f"created a line layer of {count} lifetime paths")

created a line layer of 15695 lifetime paths


In [16]:
# check structure
life_paths_gdf = geopandas.read_file(default_gdb, layer='life_paths')
life_paths_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 2252 entries, 0 to 2251
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   MusicianID               2252 non-null   int64   
 1   FIRST_Musician_Name      2251 non-null   object  
 2   FIRST_Final_Destination  2239 non-null   object  
 3   FIRST_Genre              2252 non-null   object  
 4   FIRST_All_Genres         2146 non-null   object  
 5   FIRST_Instrument         2175 non-null   object  
 6   FIRST_Free               2251 non-null   object  
 7   FIRST_Dyn                2251 non-null   object  
 8   FIRST_Dynasty_Name       705 non-null    object  
 9   Shape_Length             2252 non-null   float64 
 10  geometry                 2252 non-null   geometry
dtypes: float64(1), geometry(1), int64(1), object(8)
memory usage: 193.7+ KB


In [18]:
# ditto for this line layer, to which ArcGIS appends "FIRST_" to every field
# remove a naming convention that the PointsToLine tool adds...even though it has parameters that should stop it from doing that...

# List the field names
layer_name = "life_paths"
existing_fields = [field.name for field in arcpy.ListFields(layer_name)]

# Loop through fields. Change names and aliases.
for field in arcpy.ListFields(layer_name):
    if field.name.startswith("FIRST_"):
        new_name = field.name.replace("FIRST_", "")
        new_alias = field.aliasName.replace("FIRST_", "") if field.aliasName.startswith("FIRST_") else field.aliasName
        
        if new_name not in existing_fields:  # Avoid an error related to renaming existing fields
            arcpy.management.AlterField(layer_name, field.name, new_name, new_alias)
        else:
            print(f"Skipped {field.name} because {new_name} already exists.")

print("Field names and aliases updated.")

Field names and aliases updated.


In [19]:
# view to verify
life_paths_gdf = geopandas.read_file(default_gdb, layer='life_paths')
life_paths_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 2252 entries, 0 to 2251
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   MusicianID         2252 non-null   int64   
 1   Musician_Name      2251 non-null   object  
 2   Final_Destination  2239 non-null   object  
 3   Genre              2252 non-null   object  
 4   All_Genres         2146 non-null   object  
 5   Instrument         2175 non-null   object  
 6   Free               2251 non-null   object  
 7   Dyn                2251 non-null   object  
 8   Dynasty_Name       705 non-null    object  
 9   Shape_Length       2252 non-null   float64 
 10  geometry           2252 non-null   geometry
dtypes: float64(1), geometry(1), int64(1), object(8)
memory usage: 193.7+ KB


### Rank origins and destinations by path counts
Finally, we'll calculate the number of paths that travel through each origin and/or destination. 

In [24]:
# add path count to origins
arcpy.management.AddSpatialJoin(
    target_features="origins",
    join_features="paths",
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    field_mapping=None,
    match_option="INTERSECT",
    search_radius="5 Kilometers",
    distance_field_name="",
    permanent_join="PERMANENT_FIELDS",
    match_fields=None
)
print("added join")

## rename join count
arcpy.management.AlterField(in_table = "origins",
                            field = "Join_Count", 
                            new_field_name = "Path_Count",
                            new_field_alias = "Path_Count"
                           )
print("renamed fields")

## remove duplicate fields
layer = os.path.join(default_gdb, "origins")
field_list = arcpy.ListFields(layer)
fields_to_delete = [field.name for field in field_list if field.name.endswith('_1')]

if fields_to_delete:
    arcpy.DeleteField_management(layer, fields_to_delete)
    print(f"Deleted fields: {fields_to_delete}")
else:
    print("No fields ending with '_1' were found.")

Successfully deleted the following fields: ['SegmentID_1', 'MusicianID_1', 'Musician_Name_1', 'OriginID_1', 'DestinationID_1', 'Arrival_State_1', 'Arrival_County_1', 'Arrival_City_1', 'Surname_1', 'Depart_1', 'Arrival_1', 'Depart_Country_1', 'Depart_State_1', 'Depart_County_1', 'Depart_City_1', 'Final_Destination_1', 'Genre_1', 'Instrument_1', 'Free_1', 'Dyn_1', 'Public_1', 'Dynasty_Name_1', 'All_Genres_1', 'Afro_Cuban_Jazz_1', 'African_Jazz_1', 'Acid_Jazz_1', 'Acid_Rock_1', 'Bebop_1', 'Boogaloo_1', 'Blues_1', 'Bossa_Nova_1', 'Broadway_1', 'Brass_Band_1', 'Boogie_Woogie_1', 'Spirituals_1', 'Circus_1', 'European_Classical_1', 'Folk_Cape_Verde_1', 'Country_1', 'Disco_1', 'Doo_Wop_1', 'Early_Jazz_1', 'Easy_Listening_1', 'Free_Jazz_1', 'Funk_1', 'Folk_American_1', 'Fusion_1', 'General_Jazz_1', 'Gospel_1', 'Hard_Bop_1', 'Hip_Hop_1', 'House_1', 'Jump_Blues_1', 'Jazz_Funk_1', 'Latin_Jazz_1', 'Latin_1', 'Military_Band_1', 'Minstrelsy_1', 'Motown_1', 'Musical_Theater_1', 'New_Wave_1', 'Opera_1'

In [25]:
# add path count to destinations
arcpy.management.AddSpatialJoin(
    target_features="destinations",
    join_features="paths",
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    field_mapping=None,
    match_option="INTERSECT",
    search_radius="5 Kilometers",
    distance_field_name="",
    permanent_join="PERMANENT_FIELDS",
    match_fields=None
)
print("added join")

## rename join count
arcpy.management.AlterField(in_table = "destinations",
                            field = "Join_Count", 
                            new_field_name = "Path_Count",
                            new_field_alias = "Path_Count"
                           )
print("renamed fields")

## remove duplicate fields
layer = os.path.join(default_gdb, "destinations")
field_list = arcpy.ListFields(layer)
fields_to_delete = [field.name for field in field_list if field.name.endswith('_1')]

if fields_to_delete:
    arcpy.DeleteField_management(layer, fields_to_delete)
    print(f"Deleted fields: {fields_to_delete}")
else:
    print("No fields ending with '_1' were found.")

added join
renamed fields
Deleted fields: ['Depart_1', 'SegmentID_1', 'MusicianID_1', 'Musician_Name_1', 'OriginID_1', 'DestinationID_1', 'Arrival_State_1', 'Arrival_County_1', 'Arrival_City_1', 'Surname_1', 'Arrival_1', 'Depart_Country_1', 'Depart_State_1', 'Depart_County_1', 'Depart_City_1', 'Final_Destination_1', 'Genre_1', 'Instrument_1', 'Free_1', 'Dyn_1', 'Public_1', 'Dynasty_Name_1', 'All_Genres_1', 'Afro_Cuban_Jazz_1', 'African_Jazz_1', 'Acid_Jazz_1', 'Acid_Rock_1', 'Bebop_1', 'Boogaloo_1', 'Blues_1', 'Bossa_Nova_1', 'Broadway_1', 'Brass_Band_1', 'Boogie_Woogie_1', 'Spirituals_1', 'Circus_1', 'European_Classical_1', 'Folk_Cape_Verde_1', 'Country_1', 'Disco_1', 'Doo_Wop_1', 'Early_Jazz_1', 'Easy_Listening_1', 'Free_Jazz_1', 'Funk_1', 'Folk_American_1', 'Fusion_1', 'General_Jazz_1', 'Gospel_1', 'Hard_Bop_1', 'Hip_Hop_1', 'House_1', 'Jump_Blues_1', 'Jazz_Funk_1', 'Latin_Jazz_1', 'Latin_1', 'Military_Band_1', 'Minstrelsy_1', 'Motown_1', 'Musical_Theater_1', 'New_Wave_1', 'Opera_1',

In [ ]:
# join origins with paths
with arcpy.EnvManager(outputCoordinateSystem='PROJCRS["North_America_Albers_Equal_Area_Conic",BASEGEOGCRS["GCS_North_American_1983",DATUM["D_North_American_1983",ELLIPSOID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433]],CONVERSION["Albers",METHOD["Albers"],PARAMETER["False_Easting",0.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",-96.0],PARAMETER["Standard_Parallel_1",20.0],PARAMETER["Standard_Parallel_2",60.0],PARAMETER["Latitude_Of_Origin",40.0]],CS[Cartesian,2],AXIS["Easting (X)",east,ORDER[1]],AXIS["Northing (Y)",north,ORDER[2]],LENGTHUNIT["Meter",1.0],ID["Esri","102008"]]'):
    arcpy.analysis.SpatialJoin(
        target_features="origins",
        join_features="paths",
        out_feature_class="origins_rankedbypaths",
        join_operation="JOIN_ONE_TO_ONE",
        join_type="KEEP_ALL",
        field_mapping='MusicianID_count "MusicianID_count" true true false 4 Long 0 0,First,#,origins,MusicianID,-1,-1;Depart_Country "Depart Country" true true false 8000 Text 0 0,First,#,origins,Depart_Country,0,7999;Depart_State "Depart State" true true false 8000 Text 0 0,First,#,origins,Depart_State,0,7999;Depart_County "Depart County" true true false 8000 Text 0 0,First,#,origins,Depart_County,0,7999;Depart_City "Depart City" true true false 8000 Text 0 0,First,#,origins,Depart_City,0,7999;Afro_Cuban_Jazz "Afro-Cuban Jazz" true true false 4 Long 0 0,Sum,#,origins,Afro_Cuban_Jazz,-1,-1;African_Jazz "African Jazz" true true false 4 Long 0 0,Sum,#,origins,African_Jazz,-1,-1;Acid_Jazz "Acid Jazz" true true false 4 Long 0 0,Sum,#,origins,Acid_Jazz,-1,-1;Acid_Rock "Acid Rock" true true false 4 Long 0 0,Sum,#,origins,Acid_Rock,-1,-1;Bebop "Bebop" true true false 4 Long 0 0,Sum,#,origins,Bebop,-1,-1;Boogaloo "Boogaloo" true true false 4 Long 0 0,Sum,#,origins,Boogaloo,-1,-1;Blues "Blues" true true false 4 Long 0 0,Sum,#,origins,Blues,-1,-1;Bossa_Nova "Bossa Nova" true true false 4 Long 0 0,Sum,#,origins,Bossa_Nova,-1,-1;Broadway "Broadway" true true false 4 Long 0 0,Sum,#,origins,Broadway,-1,-1;Brass_Band "Brass Band" true true false 4 Long 0 0,Sum,#,origins,Brass_Band,-1,-1;Boogie_Woogie "Boogie-Woogie" true true false 4 Long 0 0,Sum,#,origins,Boogie_Woogie,-1,-1;Spirituals "Spirituals" true true false 4 Long 0 0,Sum,#,origins,Spirituals,-1,-1;Circus "Circus" true true false 4 Long 0 0,Sum,#,origins,Circus,-1,-1;European_Classical "European Classical" true true false 4 Long 0 0,Sum,#,origins,European_Classical,-1,-1;Folk_Cape_Verde "Folk Cape Verde" true true false 4 Long 0 0,Sum,#,origins,Folk_Cape_Verde,-1,-1;Country "Country" true true false 4 Long 0 0,Sum,#,origins,Country,-1,-1;Disco "Disco" true true false 4 Long 0 0,Sum,#,origins,Disco,-1,-1;Doo_Wop "Doo-Wop" true true false 4 Long 0 0,Sum,#,origins,Doo_Wop,-1,-1;Early_Jazz "Early Jazz" true true false 4 Long 0 0,Sum,#,origins,Early_Jazz,-1,-1;Easy_Listening "Easy Listening" true true false 4 Long 0 0,Sum,#,origins,Easy_Listening,-1,-1;Free_Jazz "Free Jazz" true true false 4 Long 0 0,Sum,#,origins,Free_Jazz,-1,-1;Funk "Funk" true true false 4 Long 0 0,Sum,#,origins,Funk,-1,-1;Folk_American "Folk American" true true false 4 Long 0 0,Sum,#,origins,Folk_American,-1,-1;Fusion "Fusion" true true false 4 Long 0 0,Sum,#,origins,Fusion,-1,-1;General_Jazz "General Jazz" true true false 4 Long 0 0,Sum,#,origins,General_Jazz,-1,-1;Gospel "Gospel" true true false 4 Long 0 0,Sum,#,origins,Gospel,-1,-1;Hard_Bop "Hard Bop" true true false 4 Long 0 0,Sum,#,origins,Hard_Bop,-1,-1;Hip_Hop "Hip Hop" true true false 4 Long 0 0,Sum,#,origins,Hip_Hop,-1,-1;House "House" true true false 4 Long 0 0,Sum,#,origins,House,-1,-1;Jump_Blues "Jump Blues" true true false 4 Long 0 0,Sum,#,origins,Jump_Blues,-1,-1;Jazz_Funk "Jazz Funk" true true false 4 Long 0 0,Sum,#,origins,Jazz_Funk,-1,-1;Latin_Jazz "Latin Jazz" true true false 4 Long 0 0,Sum,#,origins,Latin_Jazz,-1,-1;Latin "Latin" true true false 4 Long 0 0,Sum,#,origins,Latin,-1,-1;Military_Band "Military Band" true true false 4 Long 0 0,Sum,#,origins,Military_Band,-1,-1;Minstrelsy "Minstrelsy" true true false 4 Long 0 0,Sum,#,origins,Minstrelsy,-1,-1;Motown "Motown" true true false 4 Long 0 0,Sum,#,origins,Motown,-1,-1;Musical_Theater "Musical Theater" true true false 4 Long 0 0,Sum,#,origins,Musical_Theater,-1,-1;New_Wave "New Wave" true true false 4 Long 0 0,Sum,#,origins,New_Wave,-1,-1;Opera "Opera" true true false 4 Long 0 0,Sum,#,origins,Opera,-1,-1;Pop "Pop" true true false 4 Long 0 0,Sum,#,origins,Pop,-1,-1;Rhythm_and_Blues "Rhythm and Blues" true true false 4 Long 0 0,Sum,#,origins,Rhythm_and_Blues,-1,-1;Rock "Rock" true true false 4 Long 0 0,Sum,#,origins,Rock,-1,-1;Ragtime "Ragtime" true true false 4 Long 0 0,Sum,#,origins,Ragtime,-1,-1;Salsa "Salsa" true true false 4 Long 0 0,Sum,#,origins,Salsa,-1,-1;Soul_Jazz "Soul Jazz" true true false 4 Long 0 0,Sum,#,origins,Soul_Jazz,-1,-1;Smooth_Jazz "Smooth Jazz" true true false 4 Long 0 0,Sum,#,origins,Smooth_Jazz,-1,-1;Soul "Soul" true true false 4 Long 0 0,Sum,#,origins,Soul,-1,-1;Stride_Piano "Stride Piano" true true false 4 Long 0 0,Sum,#,origins,Stride_Piano,-1,-1;Swing "Swing" true true false 4 Long 0 0,Sum,#,origins,Swing,-1,-1;Vaudeville "Vaudeville" true true false 4 Long 0 0,Sum,#,origins,Vaudeville,-1,-1;World_Fusion "World Fusion" true true false 4 Long 0 0,Sum,#,origins,World_Fusion,-1,-1',
        match_option="INTERSECT",
        search_radius="5 Kilometers",
        distance_field_name="",
        match_fields=None
    )

print("ranked the origins point layer")

In [28]:
# create a simplified origins layer

## create copy
arcpy.management.Copy(
    in_data = os.path.join(default_gdb,"origins"),
    out_data = os.path.join(default_gdb, "origins_rankedbypaths"),
    data_type="FeatureClass",
    associated_data=None
)

## consolidate points
arcpy.management.CalculateGeometryAttributes(
    in_features="origins_rankedbypaths",
    geometry_property="Location POINT_COORD_NOTATION",
    length_unit="",
    area_unit="",
    coordinate_system='PROJCRS["North_America_Albers_Equal_Area_Conic",BASEGEOGCRS["GCS_North_American_1983",DATUM["D_North_American_1983",ELLIPSOID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433]],CONVERSION["Albers",METHOD["Albers"],PARAMETER["False_Easting",0.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",-96.0],PARAMETER["Standard_Parallel_1",20.0],PARAMETER["Standard_Parallel_2",60.0],PARAMETER["Latitude_Of_Origin",40.0]],CS[Cartesian,2],AXIS["Easting (X)",east,ORDER[1]],AXIS["Northing (Y)",north,ORDER[2]],LENGTHUNIT["Meter",1.0],ID["Esri","102008"]]',
    coordinate_format="SAME_AS_INPUT"
)

arcpy.management.DeleteIdentical(
    in_dataset="origins_rankedbypaths",
    fields="Location",
    xy_tolerance=None,
    z_tolerance=0,
    out_mapping_table=None
)

<Result 'origins_rankedbypaths'>

In [29]:
# view to verify 
origins_ranked = geopandas.read_file(default_gdb, layer='origins_rankedbypaths')
origins_ranked = origins_ranked.sort_values(by='Path_Count', ascending = False)
origins_ranked

,SegmentID,MusicianID,Musician_Name,OriginID,DestinationID,Arrival_State,Arrival_County,Arrival_City,Surname,Depart,...,Stride_Piano,Swing,Vaudeville,World_Fusion,LocationType,Path_Count,START_SegmentID,Depart_12,Location,geometry
125,106418211847,1064,"Henderson, Earle",nan_1064_1847,nan_1064_1821,Virginia,None,None,Strange,None,...,0,0,0,0,Origin,899,100018491895,1895.0,1436187.9060245226 -161499.58707390446,POINT (-78.69795 37.51282)
9,10019041955,100,"Ashby, Harold",New York_100_1955,Kansas City_100_1904,Missouri,Jackson,Kansas City,Ashby,None,...,0,1,0,0,Origin,855,10019041955,1955.0,1732606.4120968708 286024.8988740966,POINT (-74.00723 40.71305)
150,107718831909,1077,"Henry, Ernie",Brooklyn_1077_1909,nan_1077_1883,St. Kitts,None,None,None,None,...,0,0,0,0,Origin,834,10019041955,1955.0,1734419.5468340823 283964.74946559593,POINT (-73.991 40.69253)
1038,163818401890,1638,"Moorman, Dennis",Jersey City_1638_1890,nan_1638_1840,Virginia,None,None,Moorman,None,...,0,0,0,0,Origin,821,10019041955,1955.0,1729632.5594445465 285862.4775211867,POINT (-74.04411 40.7175)
17,100418841927,1004,"Harris, Arville",Chicago_1004_1927,St. Louis_1004_1884,Missouri,St. Louis,St. Louis,None,None,...,0,1,0,0,Origin,545,100418841927,1927.0,652111.1873558917 251134.2976104701,POINT (-87.6324 41.88323)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1423,195818801902,1958,"Russell, Felix",Bocas del Toro_1958_1902,nan_1958_1880,None,None,None,None,None,...,0,0,0,0,Origin,1,196119021917,1917.0,1600406.9547819623 -3330193.151189072,POINT (-82.2421 9.3413)
592,133619031918,1336,"Kay, Connie",nan_1336_1918,nan_1336_1903,None,None,None,James,None,...,0,0,0,0,Origin,0,None,NaN,3588645.0297909435 -2011697.4526953138,POINT (-62.18928 16.73917)
2053,28318431872,283,"Boxwill, Elinor",nan_283_1872,nan_283_1843,None,None,None,Boxwill,None,...,0,0,0,0,Origin,0,None,NaN,4390367.380450607 -3043870.9436555523,POINT (-58.97539 4.79642)
1570,20717931806,207,"Betsch, John",nan_207_1806,nan_207_1793,None,None,None,Ndiaye,None,...,0,0,0,0,Origin,0,None,NaN,1679443.1902423692 -1931529.9560708646,POINT (-79.5 22)


In [30]:
# create a simplified destinations layer

## create copy
arcpy.management.Copy(
    in_data = os.path.join(default_gdb,"destinations"),
    out_data = os.path.join(default_gdb, "destinations_rankedbypaths"),
    data_type="FeatureClass",
    associated_data=None
)

## consolidate points
arcpy.management.CalculateGeometryAttributes(
    in_features="destinations_rankedbypaths",
    geometry_property="Location POINT_COORD_NOTATION",
    length_unit="",
    area_unit="",
    coordinate_system='PROJCRS["North_America_Albers_Equal_Area_Conic",BASEGEOGCRS["GCS_North_American_1983",DATUM["D_North_American_1983",ELLIPSOID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],CS[ellipsoidal,2],AXIS["Latitude (lat)",north,ORDER[1]],AXIS["Longitude (lon)",east,ORDER[2]],ANGLEUNIT["Degree",0.0174532925199433]],CONVERSION["Albers",METHOD["Albers"],PARAMETER["False_Easting",0.0],PARAMETER["False_Northing",0.0],PARAMETER["Central_Meridian",-96.0],PARAMETER["Standard_Parallel_1",20.0],PARAMETER["Standard_Parallel_2",60.0],PARAMETER["Latitude_Of_Origin",40.0]],CS[Cartesian,2],AXIS["Easting (X)",east,ORDER[1]],AXIS["Northing (Y)",north,ORDER[2]],LENGTHUNIT["Meter",1.0],ID["Esri","102008"]]',
    coordinate_format="SAME_AS_INPUT"
)

arcpy.management.DeleteIdentical(
    in_dataset="destinations_rankedbypaths",
    fields="Location",
    xy_tolerance=None,
    z_tolerance=0,
    out_mapping_table=None
)

<Result 'destinations_rankedbypaths'>

In [31]:
# view to verify 
destinations_ranked = geopandas.read_file(default_gdb, layer='destinations_rankedbypaths')
destinations_ranked = destinations_ranked.sort_values(by='Path_Count', ascending = False)
destinations_ranked

,SegmentID,MusicianID,Musician_Name,OriginID,DestinationID,Arrival_State,Arrival_County,Arrival_City,Surname,Depart,...,Swing,Vaudeville,World_Fusion,LocationType,Path_Count,START_SegmentID,Depart_12,Depart_12_13,Location,geometry
0,100018491895,1000,"Harper, Elijah ""Buddy""",Atlanta_1000_1895,nan_1000_1849,Virginia,None,None,Blackman,None,...,0,0,0,Destination,899,100018491895,Yes,1895.0,1436187.9060245226 -161499.58707390446,POINT (-78.69795 37.51282)
41,10121919nan,1012,"Harris, Little Benny",nan_1012_nan,New York_1012_1919,New York,New York,New York,None,None,...,1,0,0,Destination,855,10019041955,None,1955.0,1732606.4120968708 286024.8988740966,POINT (-74.00723 40.71305)
1027,155619261930,1556,"McKinney, Harold",Detroit_1556_1930,Brooklyn_1556_1926,New York,Kings,Brooklyn,McKinney,None,...,0,0,0,Destination,834,10019041955,None,1955.0,1734419.5468340823 283964.74946559593,POINT (-73.991 40.69253)
1152,163818761896,1638,"Moorman, Dennis",Newark_1638_1896,Jersey City_1638_1876,New Jersey,Hudson,Jersey City,Jones,None,...,0,0,0,Destination,821,10019041955,None,1955.0,1729632.5594445465 285862.4775211867,POINT (-74.04411 40.7175)
55,101818581873,1018,"Harrison, Lawrence",Chicago_1018_1873,Chicago_1018_1858,Illinois,Cook,Chicago,Jones,None,...,0,1,0,Destination,545,100418841927,Muscogee,1927.0,652111.1873558917 251134.2976104701,POINT (-87.6324 41.88323)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1434,182418251860,1824,"Powell, John W.",Baltimore_1824_1860,nan_1824_1825,Nova Scotia,None,None,Powell,None,...,0,0,0,Destination,1,182418251860,None,1860.0,2460761.924961879 1132289.1022454007,POINT (-61.63183 45.76664)
1404,179618211846,1796,"Pichon, Walter ""Fats""",New Orleans_1796_1846,nan_1796_1821,Sonora,None,None,Peralta,None,...,0,0,0,Destination,1,179618211846,None,1846.0,-1373040.9961380041 -1096922.4122062381,POINT (-110.81071 29.69067)
4,100118211861,1001,"Harrigan, Eric",nan_1001_1861,nan_1001_1821,Anegada,None,None,None,None,...,0,0,0,Destination,0,None,None,NaN,None,POINT (NaN NaN)
330,113417801888,1134,"Hope, Elmo",nan_1134_1888,nan_1134_1780,St. Andrew,None,None,None,None,...,0,0,0,Destination,0,None,None,NaN,5422023.935796977 4026526.253366755,POINT (-2.57361 49.44778)
